In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
from torch.nn import CrossEntropyLoss
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from mygptclasses import   (My_tokenizer,
                            embedder,
                            Attention,
                            Feedforward,
                            Transformer,
                            MY_LLM,
                            Generate_text,
                            My_Dataset
                            )




In [2]:

print("CUDA available:", torch.cuda.is_available())
print("Number of CUDA GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} — {torch.cuda.get_device_name(i)}")

device = torch.device("cuda")
print("Selected device:", device)



CUDA available: True
Number of CUDA GPUs: 1
cuda:0 — NVIDIA GeForce RTX 4060 Laptop GPU
Selected device: cuda


In [ ]:
My_LLM_Config=  {
    'n_tokens':50257,
    'n_embeddings':768,
    'context_length':512,
    'n_heads': 16,
    'drop_rate': 0.1,
    'n_trans_layers':12,
    'max_tokens':64
    
    ,
    #The Maximum number of tokens you can generate.
    "stride": 128,
    "batch_size": 3,  
    "num_workers":0,
    "num_epochs":1,
    
    #These are used to configure the text generation strategy
    'top_k_for_generator':30,     #set to None if not in use
    'temperature':1.1  #set to 0 if not in use
    }


In [4]:
tokenizer=My_tokenizer()
attention_mechanism=Attention(My_LLM_Config)
feedforward_mechanism=Feedforward(My_LLM_Config)
transformer=Transformer(My_LLM_Config)
model=MY_LLM(My_LLM_Config)
#model.to(device)
text_generator=Generate_text(My_LLM_Config,model)

In [5]:
total_parameters=sum(p.numel() for p in model.parameters())
print("the total number of parameters is",{total_parameters})
   

the total number of parameters is {162694225}


In [6]:
total_size_bytes = total_parameters * 4      
total_size_mb = total_size_bytes / (1024 * 1024)    
print(f"Total size of the model: {total_size_mb:.2f} MB")

Total size of the model: 620.63 MB


In [ ]:
# import numpy as np
# from datasets import load_dataset
# from tqdm import tqdm

# tokenizer = My_tokenizer()

# # ---- settings ----
# HF_TOKEN   = ""
# MAX_TOKENS = 10_000_000_000       # placeholder: your token budget
# OUT_FILE   = "train.bin"     # placeholder: output file name
# EOT        = 50256               # GPT-2 end-of-text id
# # ------------------

# ds = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT",
#                   split="train", streaming=True, token=HF_TOKEN)

# total = 0
# with open(OUT_FILE, "wb") as out, \
#      tqdm(total=MAX_TOKENS, unit="tok", unit_scale=True, desc="Tokenizing") as pbar:
#     for doc in ds:
#         ids = tokenizer.encode(doc["text"]) + [EOT]
#         out.write(np.array(ids, dtype=np.uint16).tobytes())
#         total += len(ids)
#         pbar.update(len(ids))
#         if total >= MAX_TOKENS:
#             break

# print(f"Done: {total:,} tokens -> {OUT_FILE}")

In [8]:
# import numpy as np

# tokenizer = My_tokenizer()
# read_size = 20_000_000          # characters per piece, about 5M tokens, small RAM use

# with open("fineweb.txt", "r", encoding="utf-8") as src, open("train.bin", "wb") as out:
#     leftover = ""
#     while True:
#         piece = src.read(read_size)
#         if not piece:
#             break
#         piece = leftover + piece
#         cut = piece.rfind("\n\n")                 # cut between documents, never mid-word
#         if cut == -1:
#             leftover = piece
#             continue
#         text, leftover = piece[:cut + 2], piece[cut + 2:]
#         ids = tokenizer.encode(text)
#         out.write(np.array(ids, dtype=np.uint16).tobytes())
#     if leftover:
#         out.write(np.array(tokenizer.encode(leftover), dtype=np.uint16).tobytes())

In [9]:
class Memmap_Dataset(Dataset):
    def __init__(self, path, context_length, start, end):
        self.data = np.memmap(path, dtype=np.uint16, mode="r")
        self.start = start
        self.context_length = context_length
        self.n = (end - start - 1) // context_length

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        s = self.start + i * self.context_length
        chunk = torch.from_numpy(self.data[s:s + self.context_length + 1].astype(np.int64))
        return chunk[:-1], chunk[1:]

In [10]:
train_ratio = 0.99                                                      # changed: 99% is plenty for training, 1% is still 30M tokens for validation
data = np.memmap("train.bin", dtype=np.uint16, mode="r")                # changed: replaces tensor_encoded_training_data
split_ratio = int(train_ratio * len(data))
total_tokens = len(data)

train_data=Memmap_Dataset("train.bin",My_LLM_Config['context_length'],0,split_ratio)             # changed: replaces My_Dataset
val_data=Memmap_Dataset("train.bin",My_LLM_Config['context_length'],split_ratio,total_tokens)    # changed: replaces My_Dataset


train_loader = DataLoader(
    train_data,
    batch_size=My_LLM_Config['batch_size'],
    shuffle=True,                                                      
    num_workers=0,
   
    )

val_loader = DataLoader(
        val_data,
        batch_size=My_LLM_Config['batch_size'],
        shuffle=False,
        num_workers=0,)

In [11]:

# print("Train loader:")

# for x, y in train_loader:
#     print(x.shape, y.shape)
# print("\nValidation loader:")
# for x, y in val_loader:
#     print(x.shape, y.shape)

In [12]:
model = MY_LLM(My_LLM_Config)


def calculate_loss(model, train_loader, val_loader,  device):
    model.eval()
    model.to(device)
    train_total = 0
    val_total = 0
    train_tokens = 0
    val_tokens = 0
    loss = CrossEntropyLoss()

    with torch.no_grad():
        for i, (inputs, targets) in enumerate(train_loader):
            if i >=20:break
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            batch_loss = loss(logits.flatten(0, 1), targets.flatten())

            # Weight by token count because the last batch may be smaller.
            train_total += batch_loss.item() * targets.numel()
            train_tokens += targets.numel()

        for i, (inputs, targets) in enumerate(val_loader):
            if i >=20:break
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            batch_loss = loss(logits.flatten(0, 1), targets.flatten())

            val_total += batch_loss.item() * targets.numel()
            val_tokens += targets.numel()

    train_loss = train_total / train_tokens if train_tokens else float("nan")
    val_loss = val_total / val_tokens if val_tokens else float("nan")
    model.train()
    return train_loss, val_loss




In [ ]:
import math

def get_lr(step, max_lr, min_lr, warmup_steps, total_steps):
    # warmup: climb from near 0 up to max_lr
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    # after the end, stay at min_lr
    if step >= total_steps:
        return min_lr
    # cosine decay: smoothly shrink from max_lr down to min_lr
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

In [1]:
def train_model(model,train_loader,val_loader,optimizer,device,num_epochs,eval_freq,eval_iter,sentence_to_complete,tokenizer,
                accum_steps=8, max_lr=1e-4, min_lr=3e-5, warmup_steps=200):
    train_losses=[]
    val_losses=[]
    track_tokens_seen=[]
    tokens_seen=0
    global_step=-1
    opt_step=0   #counts real weight updates (one per accum_steps batches)
    total_opt_steps=(num_epochs*len(train_loader))//accum_steps
    best_val_loss=float("inf")
    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        for input_batch,target_batch in train_loader:
            input_batch=input_batch.to(device)
            target_batch=target_batch.to(device)
            logits=model(input_batch)
            loss=F.cross_entropy(logits.flatten(0,1),target_batch.flatten())
            (loss/accum_steps).backward()   #divide so the summed gradients average out

            tokens_seen += input_batch.numel()
            global_step+=1

            #only update the weights every accum_steps batches
            if (global_step+1) % accum_steps == 0:
                lr=get_lr(opt_step,max_lr,min_lr,warmup_steps,total_opt_steps)
                for g in optimizer.param_groups:
                    g["lr"]=lr
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step()
                optimizer.zero_grad()
                opt_step+=1

            if global_step % eval_freq ==0:
                train_loss,val_loss=calculate_loss(model, train_loader, val_loader, device)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:10d}): "
                      f"Train loss {train_loss:.3f}, "
                      f"Val loss {val_loss:.3f}, "
                      f"LR {optimizer.param_groups[0]['lr']:.2e}")

                text_generator=Generate_text(My_LLM_Config,model)

                prompt_tokens=tokenizer.encode(sentence_to_complete)
                prompt_tokens_tensor=torch.tensor(prompt_tokens, dtype=torch.long).unsqueeze(0).to(device)
                model.eval()

                generated_tokens=text_generator.forward(prompt_tokens_tensor)
                Generated_Text=text_generator.to_plain_text(generated_tokens)
                print(Generated_Text,'\n')

                if val_loss < best_val_loss:
                    best_val_loss=val_loss
                    torch.save(model.state_dict(), "my_llm_weights.pth")
                    torch.save({"model_state_dict": model.state_dict(),
                                "optimizer_state_dict": optimizer.state_dict(),}, "My_LLM_weigths_and_optimizer.pth")
                    print(f"New best model saved (Val loss {best_val_loss:.3f})",'\n','\n')
                model.train()
    return train_losses, val_losses, track_tokens_seen

In [2]:

sentence_to_complete="How are you doing today?"
torch.manual_seed(123)
num_epochs = My_LLM_Config['num_epochs']
model = MY_LLM(My_LLM_Config.//)

model.to(device)


optimizer = torch.optim.AdamW(
     model.parameters(),          
    lr=0.0004, weight_decay=0.1
)


train_losses, val_losses, tokens_seen = train_model(
    model, train_loader, val_loader, optimizer, device,num_epochs=num_epochs, eval_freq=1000, eval_iter=5,
    sentence_to_complete=sentence_to_complete, tokenizer=tokenizer
)


SyntaxError: invalid syntax (79966783.py, line 4)

In [ ]:
# checkpoint = torch.load("My_LLM_weigths_and_optimizer.pth", map_location=device)
# model = MY_LLM(My_LLM_Config)
# model.to(device)
# model.load_state_dict(checkpoint["model_state_dict"])
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.1)
# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
# model.train()

In [ ]:
# torch.manual_seed(123)
# train_losses, val_losses, tokens_seen = train_model(
#     model, train_loader, val_loader, optimizer, device,num_epochs=num_epochs, eval_freq=1000, eval_iter=5,
#     sentence_to_complete=sentence_to_complete, tokenizer=tokenizer
# )


In [ ]:
def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(
epochs_seen, val_losses, linestyle="-", label="Validation loss"
    )
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax2 = ax1.twiny()                  
    ax2.plot(tokens_seen, train_losses, alpha=0)    
    ax2.set_xlabel("Tokens seen")
    fig.tight_layout()
    plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)